In [ ]:
import os, glob, time, gc, pickle, tempfile, random, datetime
from collections import defaultdict
import numpy as np, pandas as pd
from tqdm import tqdm
from sklearn.feature_selection import mutual_info_classif
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix
)
from lightgbm import LGBMClassifier, plot_importance
from imblearn.over_sampling import RandomOverSampler
from imblearn.under_sampling import RandomUnderSampler

In [ ]:
# ===============================================================
# 1. CONFIG
# ===============================================================
# Use environment variables to avoid hard-coded sensitive paths
DATA_DIR = os.environ.get("DATA_DIR", r"F:\ML_training\data\demo3\labeled_v1")
LABEL_COL = "label"
SAVE_DIR = os.environ.get("SAVE_DIR", "saved_models")
os.makedirs(SAVE_DIR, exist_ok=True)
for sub in ["models", "reports", "predictions"]:
    os.makedirs(os.path.join(SAVE_DIR, sub), exist_ok=True)

CHUNKSIZE = 200_000
SAMPLE_FRAC_PER_CHUNK = 0.005
TOPK = 43
RANDOM_STATE = 42

tmp_dir = tempfile.mkdtemp(prefix="iot_v23_")
start_time = time.time()
print("Temporary dir:", tmp_dir)

In [ ]:
# ===============================================================
# 2. FEATURE SELECTION
# ===============================================================
print("\n[1] Feature selection (chunked sampling)...")
csv_files = sorted(glob.glob(os.path.join(DATA_DIR, "*.csv")))
if not csv_files:
    raise SystemExit(f"No CSV found in {DATA_DIR}")

sampled=[]
rows_seen=0
for fp in tqdm(csv_files, desc="Reading CSVs"):
    for c in pd.read_csv(fp, chunksize=CHUNKSIZE, low_memory=False):
        rows_seen += len(c)
        s = c.sample(frac=SAMPLE_FRAC_PER_CHUNK, random_state=RANDOM_STATE)
        sampled.append(s); del c, s; gc.collect()
data_sample = pd.concat(sampled, ignore_index=True)
print(f"Sampled {len(data_sample):,} rows from ~{rows_seen:,}")

X_s = data_sample.select_dtypes(include=[np.number]).drop(columns=[LABEL_COL], errors="ignore")
y_s = data_sample[LABEL_COL].astype(str).str.strip()
mi = mutual_info_classif(X_s.fillna(0), y_s, random_state=RANDOM_STATE)
top_idx = np.argsort(mi)[::-1][:TOPK]
selected_features = X_s.columns[top_idx].tolist()
print(f"Top {len(selected_features)} features: {selected_features[:8]} ...")
with open(os.path.join(SAVE_DIR, "selected_features.txt"), "w") as f:
    f.write("\n".join(selected_features))
del sampled, data_sample, X_s, y_s, mi; gc.collect()

In [ ]:
# ===============================================================
# 3. STREAMING LOAD AND SHUFFLE (exclude unwanted features)
# ===============================================================
print("\n[3] Streaming load and shuffle data...")

csv_files = sorted(glob.glob(os.path.join(DATA_DIR, "*.csv")))
np.random.seed(RANDOM_STATE)
np.random.shuffle(csv_files)

if not csv_files:
    raise ValueError('No CSV files found in DATA_DIR')

# Features to drop from training
FEATURES_TO_DROP = ['dst_port']
selected_features = [f for f in selected_features if f not in FEATURES_TO_DROP]
print(f"Dropped features: {FEATURES_TO_DROP}")

# Prepare list of wanted columns (features + label)
wanted_cols = set(selected_features + [LABEL_COL])

data_list = []
total_rows = 0

for fp in tqdm(csv_files, desc="Reading CSVs"):
    try:
        # Read file header to determine available columns
        file_header = pd.read_csv(fp, nrows=0).columns.tolist()
        actual_usecols = list(wanted_cols.intersection(file_header))

        # Skip file if label column missing
        if LABEL_COL not in actual_usecols:
            print(f"Skipping file {os.path.basename(fp)}: missing label '{LABEL_COL}'")
            continue

        # Read in safe columns by chunks
        for chunk in pd.read_csv(fp, chunksize=CHUNKSIZE, usecols=actual_usecols, low_memory=False):
            chunk[LABEL_COL] = chunk[LABEL_COL].astype(str).str.strip()

            # Downcast numeric dtypes to save memory
            for col in chunk.select_dtypes(include=["float64"]).columns:
                chunk[col] = pd.to_numeric(chunk[col], downcast="float")
            for col in chunk.select_dtypes(include=["int64"]).columns:
                chunk[col] = pd.to_numeric(chunk[col], downcast="integer")

            data_list.append(chunk)
            total_rows += len(chunk)

    except Exception as e:
        print(f"Warning: error reading {fp}: {e}")
        continue

print(f"Total raw rows loaded: {total_rows:,}")

if not data_list:
    raise ValueError('No data loaded')

# Concatenate and clean
data = pd.concat(data_list, ignore_index=True, copy=False)
del data_list
gc.collect()

data.fillna(0, inplace=True)

# Drop duplicates based on feature columns
print(f"Rows before removing duplicates: {len(data):,}")
feature_cols_only = [c for c in data.columns if c != LABEL_COL]
data.drop_duplicates(subset=feature_cols_only, keep='first', inplace=True)
print(f"Rows after removing duplicates:  {len(data):,}")

# Shuffle and update selected features
print("Shuffling all data...")
data = data.sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)
selected_features = [c for c in data.columns if c != LABEL_COL]
print(f"Final features list updated ({len(selected_features)} features).")
print(f"Data loaded, cleaned and shuffled. Shape: {data.shape}")

In [ ]:
# ===============================================================
# 4. LABEL ENCODING
# ===============================================================

print("\n[4] Label encoding...")
data[LABEL_COL] = data[LABEL_COL].astype(str).str.strip().str.replace(r"[\s\-]+", "_", regex=True)
unique_lbl = sorted(data[LABEL_COL].unique())
mapping = {lbl: i for i, lbl in enumerate(unique_lbl)}
data["label_encoded"] = data[LABEL_COL].map(mapping).astype("int32")
data[LABEL_COL] = data["label_encoded"]
data.drop(columns=["label_encoded"], inplace=True)
with open(os.path.join(SAVE_DIR, "label_mapping.pkl"), "wb") as f:
    pickle.dump(mapping, f)

print(f"Saving label mapping to CSV...")
mapping_csv_path = os.path.join(SAVE_DIR, "reports", "label_mapping.csv")
try:
    pd.DataFrame(list(mapping.items()), columns=["LabelName", "EncodedValue"]) \
      .sort_values("EncodedValue") \
      .to_csv(mapping_csv_path, index=False)
    print(f"Saved label mapping (csv) to {mapping_csv_path}")
except Exception as e:
    print(f"Could not save label mapping csv: {e}")
print(f"Encoded {len(mapping)} labels.")

# define classes after encoding
classes = np.array(sorted(data[LABEL_COL].unique()))
print(f"Defined {len(classes)} classes.")

# save classes
with open(os.path.join(SAVE_DIR, "classes.pkl"), "wb") as f:
    pickle.dump(classes, f)

In [ ]:
# ===============================================================
# 5. SPLIT, SCALE & NOISE INJECTION (feature + label)
# ===============================================================
print("\n[5] Splitting, scaling and noise injection...")

# split features and label
feature_names_list = [col for col in data.columns if col != LABEL_COL]
y_series = data.pop(LABEL_COL)
X_df = data
del data
gc.collect()

# train/test split (80/20)
print("Splitting train/test (80/20)...")
X_train, X_test, y_train, y_test = train_test_split(
    X_df, y_series, 
    test_size=0.2, 
    random_state=RANDOM_STATE, 
    stratify=y_series 
)

# scaling
print("Fitting standard scaler on train set...")
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

y_train = y_train.to_numpy()
y_test = y_test.to_numpy()

# feature noise
def add_gaussian_noise(X_data, noise_level=0.05):
    print(f"Injecting Gaussian noise (std={noise_level}) to training features...")
    noise = np.random.normal(0, noise_level, X_data.shape)
    return X_data + noise

X_train = add_gaussian_noise(X_train, noise_level=0.05)

# label noise: randomly swap a fraction of labels
def inject_label_noise(y_data, noise_rate=0.05):
    """Swap a fraction of labels to another random class."""
    print(f"Injecting label noise to {noise_rate*100}% of training labels...")
    n_samples = len(y_data)
    n_noise = int(n_samples * noise_rate)
    unique_classes = np.unique(y_data)
    noise_indices = np.random.choice(n_samples, n_noise, replace=False)
    swapped_count = 0
    for idx in noise_indices:
        current_label = y_data[idx]
        possible_labels = unique_classes[unique_classes != current_label]
        new_label = np.random.choice(possible_labels)
        y_data[idx] = new_label
        swapped_count += 1
    print(f"Swapped labels for {swapped_count:,} samples.")
    return y_data

# apply label noise
y_train = inject_label_noise(y_train, noise_rate=0.05)

split_summary = f"""
Train features: {X_train.shape} (with noise)
Train labels: {y_train.shape}
Test features: {X_test.shape} (clean)
Test labels: {y_test.shape}
"""
print(split_summary)

# saving artifacts
print("Saving artifacts...")
with open(os.path.join(SAVE_DIR, "models", "scaler.pkl"), "wb") as f:
    pickle.dump(scaler, f)
with open(os.path.join(SAVE_DIR, "models", "feature_names.pkl"), "wb") as f:
    pickle.dump(feature_names_list, f)
inv_mapping = {v: k for k, v in mapping.items()}
with open(os.path.join(SAVE_DIR, "models", "label_mapping_inverse.pkl"), "wb") as f:
    pickle.dump(inv_mapping, f)

print("Data preparation complete.")

In [ ]:
# ===============================================================
# 6. TRAIN MULTI-MODEL & SAVE LightGBM
# ===============================================================
print("\n[6] Training and comparing multiple models...")
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from xgboost import XGBClassifier
import matplotlib.pyplot as plt

models_to_run = {
    "LightGBM": LGBMClassifier(
        boosting_type="gbdt", 
        objective="multiclass",
        num_class=len(classes), 
        n_estimators=1000, 
        learning_rate=0.01, 
        num_leaves=7,
        max_depth=5,
        reg_alpha=5.0,
        reg_lambda=10.0,
        subsample=0.8,
        colsample_bytree=0.5,
        min_child_samples=3000,
        class_weight='balanced',
        random_state=RANDOM_STATE,
        n_jobs=-1,
        verbose=-1
    ),

    "RandomForest": RandomForestClassifier(
        n_estimators=200, 
        max_depth=15,
        max_features='sqrt',
        min_samples_leaf=5,
        class_weight='balanced',
        n_jobs=-1, 
        random_state=RANDOM_STATE
    ),

    "XGBoost": XGBClassifier(
        n_estimators=1000,
        learning_rate=0.01,
        max_depth=5,
        reg_alpha=5.0,
        reg_lambda=10.0,
        subsample=0.8,
        colsample_bytree=0.5,
        min_child_weight=30,
        class_weight='balanced',
        n_jobs=-1, 
        objective="multi:softmax",
        num_class=len(classes), 
        use_label_encoder=False,
        eval_metric='mlogloss',
        random_state=RANDOM_STATE
    ),

    "DecisionTree": DecisionTreeClassifier(
        max_depth=10,
        min_samples_leaf=10,
        class_weight='balanced',
        random_state=RANDOM_STATE
    ),

    "LogisticRegression": LogisticRegression(
        max_iter=500,
        C=0.1,
        penalty='l2',
        class_weight='balanced',
        n_jobs=-1, 
        random_state=RANDOM_STATE,
    ),

    "NaiveBayes": GaussianNB()    
}

results = []
feature_names = feature_names_list

for name, model in models_to_run.items():
    print(f"\n>>> Training {name} ...")
    t0 = time.time()

    if name == "LightGBM":
        model.fit(X_train, y_train, feature_name=feature_names, categorical_feature=None)
    else:
        model.fit(X_train, y_train)

    y_pred = model.predict(X_test)

    acc = accuracy_score(y_test, y_pred)
    pre = precision_score(y_test, y_pred, average="macro", zero_division=0)
    rec = recall_score(y_test, y_pred, average="macro", zero_division=0)
    f1 = f1_score(y_test, y_pred, average="macro", zero_division=0)
    runtime = round(time.time() - t0, 2)

    results.append({
        "Model": name,
        "Accuracy": acc,
        "Precision": pre,
        "Recall": rec,
        "F1_macro": f1,
        "Runtime_sec": runtime,
        "Train_Timestamp": datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    })

    if name == "LightGBM":
        model_path = os.path.join(SAVE_DIR, "models", "LightGBM_model.pkl")
        with open(model_path, "wb") as f:
            pickle.dump({
                "model": model,
                "scaler": scaler,
                "features": feature_names,
                "mapping": mapping
            }, f)
        print(f"Saved LightGBM model to {model_path}")

        # confusion matrix
        cm = confusion_matrix(y_test, y_pred, normalize="true")
        cm_path = os.path.join(SAVE_DIR, "reports", "conf_matrix_LightGBM.csv")
        pd.DataFrame(cm).to_csv(cm_path, index=False)
        print(f"Saved LightGBM confusion matrix to {cm_path}")

        # classification report
        print("Generating classification report for LightGBM...")
        try:
            inv_mapping = {v: k for k, v in mapping.items()}
            target_names = [inv_mapping[i] for i in range(len(mapping))]
            report = classification_report(y_test, y_pred, target_names=target_names, zero_division=0)
            report_path = os.path.join(SAVE_DIR, "reports", "classification_report_LGBM.txt")
            with open(report_path, "w", encoding="utf-8") as f:
                f.write("=== CLASSIFICATION REPORT (LightGBM) ===\n")
                f.write(f"Saved at: {datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n\n")
                f.write(report)
            print(f"Saved classification report to {report_path}")
            print(report)
        except Exception as e:
            print(f"Could not generate classification report: {e}")

        # feature importance plot
        print("Generating feature importance for LightGBM...")
        try:
            fig, ax = plt.subplots(figsize=(12, 10))
            plot_importance(
                model, 
                ax=ax, 
                max_num_features=20, 
                importance_type='gain',
                title='LightGBM Feature Importance (Top 20 by Gain)',
            )
            plt.tight_layout()
            img_path = os.path.join(SAVE_DIR, "reports", "feature_importance_LGBM.png")
            plt.savefig(img_path, dpi=150)
            plt.close(fig)
            print(f"Saved feature importance plot to {img_path}")
        except Exception as e:
            print(f"Could not generate feature importance plot: {e}")

        # save feature importance data
        print("Saving feature importance data to CSV...")
        try:
            importances_gain = model.booster_.feature_importance(importance_type='gain')
            feat_names = model.feature_name_
            fi_df = pd.DataFrame({
                'feature': feat_names,
                'importance_gain': importances_gain
            }).sort_values(by='importance_gain', ascending=False)
            fi_csv_path = os.path.join(SAVE_DIR, "reports", "feature_importance_data.csv")
            fi_df.to_csv(fi_csv_path, index=False)
            print(f"Saved feature importance data to {fi_csv_path}")
        except Exception as e:
            print(f"Could not save feature importance data: {e}")

        def get_binary_metrics_per_class(y_true, y_pred, mapping):
            """
            Compute TP, TN, FP, FN and binary metrics per class (one-vs-rest).
            """
            classes_ids = sorted(mapping.values())
            inv_map = {v: k for k, v in mapping.items()}
            metrics_list = []
            cm = confusion_matrix(y_true, y_pred, labels=classes_ids)
            for i, class_id in enumerate(classes_ids):
                label_name = inv_map[class_id]
                TP = cm[i, i]
                FP = cm[:, i].sum() - TP
                FN = cm[i, :].sum() - TP
                TN = cm.sum() - (TP + FP + FN)
                eps = 1e-7
                Accuracy = (TP + TN) / (TP + TN + FP + FN + eps)
                Precision = TP / (TP + FP + eps)
                Recall = TP / (TP + FN + eps)
                Specificity = TN / (TN + FP + eps)
                F1 = 2 * (Precision * Recall) / (Precision + Recall + eps)
                metrics_list.append({
                    "Label_ID": class_id,
                    "Label_Name": label_name,
                    "TP": TP,
                    "TN": TN,
                    "FP": FP,
                    "FN": FN,
                    "Accuracy": round(Accuracy, 4),
                    "Precision": round(Precision, 4),
                    "Recall": round(Recall, 4),
                    "Specificity": round(Specificity, 4),
                    "F1_Score": round(F1, 4)
                })
            return pd.DataFrame(metrics_list)

        print("Calculating binary (one-vs-rest) metrics for LightGBM...")
        binary_df = get_binary_metrics_per_class(y_test, y_pred, mapping)
        print("Binary Metrics Breakdown (sample):")
        print(binary_df.sort_values("F1_Score").head(10).to_string(index=False))
        binary_report_path = os.path.join(SAVE_DIR, "reports", "binary_metrics_per_class.csv")
        binary_df.to_csv(binary_report_path, index=False)
        print(f"Saved binary metrics report to: {binary_report_path}")

    print(f"{name}: F1={f1:.4f}, Acc={acc:.4f}, time={runtime}s")

In [ ]:
try:
    res_df = pd.DataFrame(results)
    display(res_df)
    res_df.to_csv("partial_results_backup.csv", index=False)
    print("Partial results saved to partial_results_backup.csv")
except NameError:
    print("No results available.")

In [ ]:
# ===============================================================
# 7. SUMMARY
# ===============================================================
res = pd.DataFrame(results).sort_values("F1_macro", ascending=False)
res_path = os.path.join(SAVE_DIR, "reports", "model_comparison_summary.csv")
res.to_csv(res_path, index=False)
print("Summary:\n", res)
print(f"Saved model comparison summary to: {res_path}")

end_time = time.time()
total_runtime = round(end_time - start_time, 2)
print(f"Pipeline started at: {datetime.datetime.fromtimestamp(start_time)}")
print(f"Finished at: {datetime.datetime.fromtimestamp(end_time)}")
print(f"Total runtime (min): {total_runtime/60:.2f}")

# save pipeline summary
pipeline_summary_path = os.path.join(SAVE_DIR, "reports", "pipeline_summary.txt")
try:
    with open(pipeline_summary_path, "w", encoding="utf-8") as f:
        f.write("=== PIPELINE SUMMARY ===\n")
        f.write(f"Pipeline Start: {datetime.datetime.fromtimestamp(start_time)}\n")
        f.write(f"Pipeline End: {datetime.datetime.fromtimestamp(end_time)}\n")
        f.write(f"Total Runtime (sec): {total_runtime}\n")
        f.write(f"Total Runtime (min): {total_runtime/60:.2f}\n")
        f.write("\n--- Data Split ---\n")
        f.write(split_summary)
        best_model_stats = res.iloc[0]
        f.write("\n--- Best Model Stats ---\n")
        f.write(f"Model: {best_model_stats['Model']}\n")
        f.write(f"F1_macro: {best_model_stats['F1_macro']:.6f}\n")
        f.write(f"Accuracy: {best_model_stats['Accuracy']:.6f}\n")
        f.write(f"Precision: {best_model_stats['Precision']:.6f}\n")
        f.write(f"Recall: {best_model_stats['Recall']:.6f}\n")
        f.write(f"Runtime_sec: {best_model_stats['Runtime_sec']}\n")
    print(f"Saved pipeline summary to: {pipeline_summary_path}")
except Exception as e:
    print(f"Could not save pipeline summary: {e}")

print("Training pipeline completed.")